# Exploración de la reconstrucción de los autoencoders

Plantilla del change `ae-reconstruccion` (tarea 8.4). **Nivel 1: sin TensorFlow.**
Lee `reconstruction.parquet`, `diagnostics.parquet` y las métricas del run de AE
desde el estado de MLflow (`analysis/load.py`, sin duplicar la carga).

Diagnósticos de aplanamiento: razón de varianza, pendiente real vs reconstruido,
error por nivel de concentración, y comparación contra `ref-mean` y `ref-pca-d*`.

In [ ]:
import sys, pathlib
# Raíz de importación del pipeline: el notebook vive en notebooks/ (o en la raíz del repo).
_here = pathlib.Path.cwd()
PIPELINE = next(p for p in (_here / "pipeline", _here.parent / "pipeline") if p.exists())
sys.path.insert(0, str(PIPELINE))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from analysis import load, plots_ae
from features.build import load_validated
from ae import data as aed
from ae import dmd as dmdm
from ae import metrics as aem

BUCKET_DATA = "sinca-data"
# Credenciales: el rol del Space (sinca-dev-role) las provee; no escribir claves aquí.

## 1. Estado de MLflow y runs de autoencoder

In [ ]:
state = load.ensure_state()  # S3 real (sinca-mlflow/_mlflow); o load.ensure_state(data_dir=<fixture>)
runs = load.list_runs(state)
ae_runs = load.ae_runs(runs)
refs = [r for r in runs if load.is_reference_run(r)]
print("AE:", [r["run_name"] for r in ae_runs])
print("Referencias:", [r["run_name"] for r in refs])

## 2. Elegir un run y cargar sus artefactos

In [ ]:
RUN_NAME = ae_runs[-1]["run_name"]  # cambiar a mano para comparar otro run
run = next(r for r in ae_runs if r["run_name"] == RUN_NAME)
arts = load.load_artifacts(state, run["run_id"])
recon, diag = arts["reconstruction.parquet"], arts["diagnostics.parquet"]
print("compuerta:", run["metrics"].get("gate_pass"), "| supera PCA:", run["metrics"].get("gate_beats_pca"))

## 3. Real vs reconstruido con zoom a un episodio

In [ ]:
serie = "Nielol|pm25"
fig = plots_ae.reconstruction_figure(recon, serie, start="2026-07-01", end="2026-07-15")
fig.show()

## 4. Aplanamiento y error por nivel de concentración

In [ ]:
display(diag.pivot_table(index="serie", columns="metric", values="value").round(3))
plots_ae.flattening_figure(diag, "test").show()
plots_ae.error_by_level_figure(diag, "test").show()

## 5. Contra las referencias y ablación de z_aux

In [ ]:
rows = [{"run": run["run_name"], **load.ae_metric(run, "recon_rmse_mean_")}]
rows += [{"run": r["run_name"], "test": r["metrics"].get("recon_rmse_mean_test"), "valid": r["metrics"].get("recon_rmse_mean_valid")} for r in refs]
display(pd.DataFrame(rows))
print("Δ RMSE al anular z_aux (test):", run["metrics"].get("ablation_zaux_delta_rmse_test"))